# Learning with Less
## News Classification in Four Nigerian Languages

One comparison repeated separately for Yorùbá, Igbo, Hausa and Nigerian Pidgin. We test word versus character TF-IDF with the same classifier, using progressively larger training sets.

The scripts have been executed and the attached results verified. This notebook is supplied unexecuted so you can reproduce the results yourself. It reuses the same model settings and filtering rules as your completed Yorùbá experiment.

Data: [MasakhaNEWS](https://huggingface.co/datasets/masakhane/masakhanews), [Adelani et al. (2023)](https://aclanthology.org/2023.ijcnlp-main.10/).

### 1. Install dependencies
After installing, restart the session if prompted and continue below without repeating this cell. Colab may report a NumPy conflict with its preinstalled numba; these experiments do not use numba. The pinned versions match the reproduced Yorùbá run.

In [ ]:
%pip -q install numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 matplotlib==3.10.8

### 2. Create the experiment scripts
This cell writes the shared experiment code into a new folder. Each language gets its own data and results folders. Expand the code to inspect it; the readable scripts are also included in the ZIP.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
PROJECT = Path.cwd() / 'four_language_study'
PROJECT.mkdir(exist_ok=True)
(PROJECT / 'data_utils.py').write_text('"""First experiment: audit Yorùbá data and evaluate two fixed baselines on dev.\n\nRun: python start_project.py\nThis script deliberately does not download or evaluate the final test split.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport platform\nimport time\nimport unicodedata\nimport urllib.request\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, f1_score\nfrom sklearn.pipeline import Pipeline\n\nROOT = Path(__file__).resolve().parent\nREVISION = \'fa3b5fff8a91d187bf0c5900a39c4271d08cf7fe\'\nBASE = f\'https://huggingface.co/datasets/masakhane/masakhanews/resolve/{REVISION}/data/yor/\'\n\ndef normalise(text):\n    return \' \'.join(unicodedata.normalize(\'NFC\', str(text)).lower().split())\n\ndef word_tokens(text):\n    # Preserve Yorùbá combining tone marks within words. Python\'s default\n    # word regex can split at combining marks, even after NFC normalisation.\n    words, current = [], []\n    for c in text:\n        if unicodedata.category(c)[0] in \'LMN\':\n            current.append(c)\n        elif current:\n            words.append(\'\'.join(current))\n            current = []\n    if current:\n        words.append(\'\'.join(current))\n    return words\n\ndef load_split(name):\n    path = ROOT / \'data\' / f\'{name}.tsv\'\n    path.parent.mkdir(exist_ok=True)\n    if not path.exists():\n        with urllib.request.urlopen(BASE + path.name, timeout=90) as response:\n            path.write_bytes(response.read())\n    df = pd.read_csv(path, sep=\'\\t\').fillna(\'\')\n    required = {\'text\', \'headline\', \'category\', \'url\'}\n    assert required.issubset(df.columns), df.columns\n    df[\'input\'] = df[\'text\'].map(normalise)\n    return df, hashlib.sha256(path.read_bytes()).hexdigest()\n', encoding='utf-8')
(PROJECT / 'study_engine.py').write_text('"""Frozen comparison: 2 representations x 3 budgets, matched stratified subsets."""\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport json, time, warnings, hashlib\nfrom datetime import datetime, timezone\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix\nfrom sklearn.exceptions import ConvergenceWarning\nfrom data_utils import load_split, word_tokens, REVISION\n\nROOT=Path(__file__).resolve().parent\nOUT=ROOT/\'results\'\nOUT.mkdir(exist_ok=True)\nSEEDS=[11,22,33,44,55]\nPROTOCOL={\n \'recorded_utc\':datetime.now(timezone.utc).isoformat(),\n \'data_revision\':REVISION,\'seeds\':SEEDS,\'fractions\':[0.25,0.5,1.0],\n \'inputs\':\'NFC/lowercased article bodies only; preserve combining marks; no headlines or URLs\',\n \'word_features\':\'word unigrams and bigrams; Unicode letter/mark/number tokenizer\',\n \'character_features\':\'character 3-5 grams, including spaces\',\n \'shared_tfidf\':{\'min_df\':2,\'max_features\':50000,\'sublinear_tf\':True,\'norm\':\'l2\'},\n \'classifier\':{\'type\':\'LogisticRegression\',\'C\':1.0,\'solver\':\'lbfgs\',\'max_iter\':1000},\n \'sampling\':\'For each seed, permute each training class; take floor(fraction * class size) from each class. Nested matched subsets. Full training set fitted once per approach.\',\n \'deduplication\':\'Remove empty bodies; greedy first-occurrence dedup by normalized exact body, nonempty URL, or >=0.80 Jaccard of word 5-gram sets. Training is priority, then dev, then test. Do not use labels for overlap filtering.\',\n \'evaluation\':\'Official test split after prespecified overlap filtering; no model selection or tuning from test results. Macro-F1 primary; accuracy secondary. Mean and sample SD over five subsets at 25/50%; 100% single deterministic fit.\',\n \'scope\':\'Exploratory comparison on one corpus; no significance or state-of-the-art claim; subset SD is not a test-population confidence interval.\'}\n\nclass DuplicateIndex:\n    def __init__(self):\n        self.texts={}; self.urls={}; self.shingles=[]; self.refs=[]; self.postings=defaultdict(list)\n    def check_add(self,text,url,ref):\n        if not text:return {\'ref\':ref,\'reason\':\'empty\'}\n        if text in self.texts:return {\'ref\':ref,\'matches\':self.texts[text],\'reason\':\'exact_text\'}\n        if url and url in self.urls:return {\'ref\':ref,\'matches\':self.urls[url],\'reason\':\'url\'}\n        tokens=word_tokens(text)\n        shingles=set(zip(*(tokens[i:] for i in range(5))))\n        counts=Counter()\n        for s in shingles:counts.update(self.postings.get(s,[]))\n        for i,intersection in counts.items():\n            jaccard=intersection/(len(shingles)+len(self.shingles[i])-intersection)\n            if jaccard>=0.8:return {\'ref\':ref,\'matches\':self.refs[i],\'reason\':\'near_duplicate\',\'jaccard\':jaccard}\n        i=len(self.refs);self.refs.append(ref);self.shingles.append(shingles)\n        self.texts[text]=ref\n        if url:self.urls[url]=ref\n        for s in shingles:self.postings[s].append(i)\n        return None\n\ndef main():\n    # Write protocol before loading/evaluating test. No alterations based on outcomes.\n    (OUT/\'protocol.json\').write_text(json.dumps(PROTOCOL,indent=2))\n    idx=DuplicateIndex();frames={};audit={\'original\':{},\'retained\':{},\'sha256\':{},\'removed\':[]}\n    for split in [\'train\',\'dev\',\'test\']:\n        df,sha=load_split(split);audit[\'original\'][split]=len(df);audit[\'sha256\'][split]=sha\n        keep=[]\n        for rowid,row in df.iterrows():\n            removal=idx.check_add(row[\'input\'],row[\'url\'],f\'{split}:{rowid}\')\n            if removal:audit[\'removed\'].append(removal)\n            else:keep.append(rowid)\n        frames[split]=df.loc[keep].copy();audit[\'retained\'][split]=len(keep)\n        print(\'AUDIT\',split,len(df),\'->\',len(keep),flush=True)\n    (OUT/\'audit.json\').write_text(json.dumps(audit,indent=2))\n    train=frames[\'train\'];test=frames[\'test\'];labels=sorted(train[\'category\'].unique())\n    assert set(test[\'category\'])==set(labels)\n    counts={split:{str(k):int(v) for k,v in df[\'category\'].value_counts().sort_index().items()} for split,df in frames.items()}\n    (OUT/\'class_counts.json\').write_text(json.dumps(counts,indent=2))\n    majority=train[\'category\'].value_counts().idxmax()\n    (OUT/\'majority_baseline.json\').write_text(json.dumps({\'category\':majority,\n       \'macro_f1\':f1_score(test[\'category\'],[majority]*len(test),average=\'macro\',labels=labels,zero_division=0),\n       \'accuracy\':accuracy_score(test[\'category\'],[majority]*len(test))},indent=2))\n    rows=[];allpred=[];samples={}\n    for seed in SEEDS:\n        rng=np.random.default_rng(seed)\n        permutations={c:rng.permutation(train.index[train[\'category\']==c].to_numpy()) for c in labels}\n        for frac in [0.25,0.5,1.0]:\n            if frac==1 and seed!=SEEDS[0]:continue\n            ids=np.concatenate([permutations[c][:max(1,int(np.floor(frac*len(permutations[c]))))] for c in labels])\n            ids=np.sort(ids);subset=train.loc[ids]\n            samples[f\'{seed}_{frac}\']=[int(i) for i in ids]\n            for approach in [\'Word\',\'Character\']:\n                opts=dict(min_df=2,max_features=50000,sublinear_tf=True,lowercase=False)\n                vec=TfidfVectorizer(tokenizer=word_tokens,token_pattern=None,ngram_range=(1,2),**opts) if approach==\'Word\' else TfidfVectorizer(analyzer=\'char\',ngram_range=(3,5),**opts)\n                model=Pipeline([(\'tfidf\',vec),(\'classifier\',LogisticRegression(C=1.,solver=\'lbfgs\',max_iter=1000))])\n                start=time.perf_counter()\n                with warnings.catch_warnings(record=True) as caught:\n                    warnings.simplefilter(\'always\',ConvergenceWarning)\n                    model.fit(subset[\'input\'],subset[\'category\'])\n                seconds=time.perf_counter()-start\n                n_warnings=sum(issubclass(w.category,ConvergenceWarning) for w in caught)\n                if n_warnings:raise RuntimeError(\'Convergence warning: results not ready to report.\')\n                predicted=model.predict(test[\'input\'])\n                result={\'approach\':approach,\'fraction\':frac,\'seed\':seed,\'n_train\':len(subset),\'n_test\':len(test),\n                    \'macro_f1\':f1_score(test[\'category\'],predicted,average=\'macro\',labels=labels,zero_division=0),\n                    \'accuracy\':accuracy_score(test[\'category\'],predicted),\'fit_seconds\':seconds,\'features\':len(vec.vocabulary_)}\n                rows.append(result)\n                print(\'RESULT\',result,flush=True)\n                for rowid,true,pred in zip(test.index,test[\'category\'],predicted):allpred.append({\'approach\':approach,\'fraction\':frac,\'seed\':seed,\'row_id\':int(rowid),\'true\':true,\'predicted\':pred})\n                pd.DataFrame(rows).to_csv(OUT/\'runs.csv\',index=False)\n                if frac==1:\n                    (OUT/f\'{approach.lower()}_classification_report.json\').write_text(json.dumps(classification_report(test[\'category\'],predicted,labels=labels,output_dict=True,zero_division=0),indent=2))\n                    pd.DataFrame(confusion_matrix(test[\'category\'],predicted,labels=labels),index=labels,columns=labels).to_csv(OUT/f\'{approach.lower()}_confusion.csv\')\n    pd.DataFrame(allpred).to_csv(OUT/\'test_predictions.csv\',index=False)\n    (OUT/\'training_row_ids.json\').write_text(json.dumps(samples))\n    runs=pd.DataFrame(rows)\n    summary=runs.groupby([\'approach\',\'fraction\',\'n_train\']).agg(n_runs=(\'seed\',\'count\'),macro_f1_mean=(\'macro_f1\',\'mean\'),macro_f1_sd=(\'macro_f1\',\'std\'),accuracy_mean=(\'accuracy\',\'mean\'),accuracy_sd=(\'accuracy\',\'std\')).reset_index()\n    summary.to_csv(OUT/\'summary.csv\',index=False)\n    print(\'SUMMARY\\n\'+summary.to_string(index=False),flush=True)\n    assert len(rows)==22 and len(allpred)==22*len(test)\n\nif __name__==\'__main__\':main()\n', encoding='utf-8')
(PROJECT / 'run_language.py').write_text('"""Use the unchanged Yorùbá experiment with a separate dataset per language."""\nfrom pathlib import Path\nimport argparse\nimport data_utils\nimport study_engine\n\nLANGUAGES={\'yor\':\'Yorùbá\',\'ibo\':\'Igbo\',\'hau\':\'Hausa\',\'pcm\':\'Nigerian Pidgin\'}\n\ndef run_language(code):\n    if code not in LANGUAGES:raise ValueError(code)\n    root=Path(__file__).resolve().parent/\'languages\'/code\n    root.mkdir(parents=True,exist_ok=True)\n    data_utils.ROOT=root\n    data_utils.BASE=f\'https://huggingface.co/datasets/masakhane/masakhanews/resolve/{data_utils.REVISION}/data/{code}/\'\n    study_engine.ROOT=root\n    study_engine.OUT=root/\'results\'\n    study_engine.OUT.mkdir(exist_ok=True)\n    study_engine.PROTOCOL.update({\'language_code\':code,\'language\':LANGUAGES[code],\n        \'study_history\':\'Extension selected after observing the Yorùbá result. Model settings, sampling and overlap checks unchanged. Separate monolingual models; no pooled multilingual model.\'})\n    print(\'LANGUAGE\',LANGUAGES[code],flush=True)\n    study_engine.main()\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'language\',choices=LANGUAGES)\n    run_language(parser.parse_args().language)\n', encoding='utf-8')
(PROJECT / 'summarize_results.py').write_text("from pathlib import Path\nimport json\nimport pandas as pd\nimport numpy as np\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n\nROOT=Path(__file__).resolve().parent\nLANGS={'yor':'Yorùbá','ibo':'Igbo','hau':'Hausa','pcm':'Nigerian Pidgin'}\nFIG=ROOT/'figures';FIG.mkdir(exist_ok=True)\nframes=[];audits={};counts={}\nfor code,name in LANGS.items():\n    folder=ROOT/'languages'/code/'results'\n    d=pd.read_csv(folder/'summary.csv');d['language']=name;d['code']=code;frames.append(d)\n    audits[code]=json.loads((folder/'audit.json').read_text())\n    counts[code]=json.loads((folder/'class_counts.json').read_text())\nsummary=pd.concat(frames,ignore_index=True)\nsummary.to_csv(ROOT/'combined_summary.csv',index=False)\ncomparison=summary.pivot(index=['code','language','fraction','n_train'],columns='approach',values='macro_f1_mean').reset_index()\ncomparison['character_minus_word_points']=100*(comparison.Character-comparison.Word)\ncomparison.to_csv(ROOT/'comparison.csv',index=False)\nprint(comparison.to_string(index=False),flush=True)\n\nplt.rcParams.update({'font.family':'DejaVu Sans','font.size':12,'axes.spines.top':False,'axes.spines.right':False})\nfig,axes=plt.subplots(2,2,figsize=(12,9),layout='constrained')\nfor ax,(code,name) in zip(axes.flat,LANGS.items()):\n    part=summary[summary.code==code]\n    for approach,color,marker in [('Word','#176B8C','o'),('Character','#C45B29','s')]:\n        d=part[part.approach==approach].sort_values('fraction')\n        ax.plot(d.fraction*100,d.macro_f1_mean*100,color=color,marker=marker,lw=2.3,label=approach)\n        e=d[d.n_runs>1]\n        ax.errorbar(e.fraction*100,e.macro_f1_mean*100,yerr=e.macro_f1_sd*100,fmt='none',ecolor=color,capsize=4)\n    ntest=audits[code]['retained']['test'];nclass=len(counts[code]['train'])\n    ax.set(title=f'{name} | {nclass} categories | {ntest} test articles',ylim=(45,95),xticks=[25,50,100],xlabel='Training budget (%)',ylabel='Test macro-F1 (0–100)')\n    sizes=part[part.approach=='Word'].sort_values('fraction').n_train.tolist()\n    ax.set_xticklabels([f'{f}%\\n(n={n:,})' for f,n in zip([25,50,100],sizes)])\n    ax.grid(axis='y',alpha=.17)\nhandles,labels=axes[0,0].get_legend_handles_labels()\nfig.legend(handles,labels,loc='outside upper center',ncol=2,frameon=False)\nfor ext in ['png','pdf']:fig.savefig(FIG/f'four_language_learning_curves.{ext}',dpi=230)\nplt.close(fig)\n\n", encoding='utf-8')


### 3. Select languages and run
Default: reproduce all four. These are separate models, not a jointly trained multilingual model. The same rules remove overlapping articles before evaluation. Scores may take several minutes to appear while datasets download and checks run. Use CPU; no GPU is required.

To run only the additional languages, change the list to `["ibo", "hau", "pcm"]`; the combined plot needs results for all four, so the default is recommended in a fresh notebook.

In [ ]:
LANGUAGES = ['yor', 'ibo', 'hau', 'pcm']
env = dict(os.environ, OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1')
for language in LANGUAGES:
    subprocess.run([sys.executable, '-u', str(PROJECT/'run_language.py'), language], env=env, check=True)

### 4. Compare and plot
Within each language, compare Word and Character at each training budget. Different languages have different dataset sizes, categories and test sets, so do not rank their difficulty by their raw scores. Bars show training-subset SD, not statistical significance.

In [ ]:
subprocess.run([sys.executable, str(PROJECT/'summarize_results.py')], env=env, check=True)
import pandas as pd
summary = pd.read_csv(PROJECT/'combined_summary.csv')
summary[['language','approach','n_train','n_runs','macro_f1_mean','macro_f1_sd']].assign(macro_f1_mean=lambda d:100*d.macro_f1_mean, macro_f1_sd=lambda d:100*d.macro_f1_sd)

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(PROJECT/'figures/four_language_learning_curves.png')))

### 5. Save your work
Download the notebook with its outputs through Colab's Commands menu. This cell separately downloads scripts, results, hashes and figures as a ZIP. Raw downloaded articles are excluded to keep the backup small; the scripts can fetch them again.

In [ ]:
import zipfile
archive = PROJECT.parent/'Four_Language_Results_Backup.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in PROJECT.rglob('*'):
        rel = path.relative_to(PROJECT)
        if path.is_file() and 'data' not in rel.parts and '__pycache__' not in rel.parts:
            z.write(path,rel)
from google.colab import files
files.download(str(archive))

### Reference results from the verified run

# Results: Four Nigerian Languages

Character features had higher mean macro-F1 in all 12 language-budget comparisons. Differences range from 1.11 to 3.08 macro-F1 points. These are descriptive differences; statistical significance is untested.

Scores use a 0–100 scale. ± is sample SD over five matched training subsets. Full-data results use one fit per representation.

| Language | Train articles | Budget | Word macro-F1 | Character macro-F1 |
|---|---:|---:|---:|---:|
| Yorùbá | 356 | 25% | 78.26 ± 1.08 | 81.25 ± 1.11 |
| Yorùbá | 716 | 50% | 82.67 ± 0.83 | 85.76 ± 0.65 |
| Yorùbá | 1,433 | 100% | 86.79 | 87.90 |
| Igbo | 321 | 25% | 57.27 ± 1.53 | 58.82 ± 2.02 |
| Igbo | 647 | 50% | 63.63 ± 0.84 | 65.73 ± 1.43 |
| Igbo | 1,295 | 100% | 69.94 | 72.63 |
| Hausa | 551 | 25% | 76.73 ± 1.67 | 79.51 ± 1.59 |
| Hausa | 1,105 | 50% | 81.63 ± 0.24 | 83.52 ± 0.68 |
| Hausa | 2,216 | 100% | 85.50 | 86.67 |
| Nigerian Pidgin | 262 | 25% | 64.15 ± 0.73 | 66.47 ± 1.06 |
| Nigerian Pidgin | 528 | 50% | 71.82 ± 0.60 | 73.45 ± 1.53 |
| Nigerian Pidgin | 1,058 | 100% | 82.75 | 84.86 |

## Data retained after the same overlap checks

| Language | Train before → after | Dev before → after | Test before → after | Categories |
|---|---:|---:|---:|---:|
| Yorùbá | 1433 → 1433 | 206 → 206 | 411 → 411 | 5 |
| Igbo | 1356 → 1295 | 194 → 174 | 390 → 361 | 6 |
| Hausa | 2219 → 2216 | 317 → 316 | 637 → 633 | 7 |
| Nigerian Pidgin | 1060 → 1058 | 152 → 152 | 305 → 300 | 5 |

A total of 125 rows were removed across all three partitions: 110 for Igbo, 8 for Hausa and 7 for Nigerian Pidgin. These are rows removed, not 125 independent news stories. No Yorùbá rows were removed.

The test sets and category sets differ, so compare representations within each language. Scores must not be used to rank languages by inherent difficulty. Raw accuracy can obscure category imbalance: the abstract and curves therefore use macro-F1.

Both approaches improve with more training data in the mean results. The character advantage does not consistently shrink as data increases across all four languages. Do not carry the Yorùbá-only claim of a narrowing gap into a universal four-language conclusion.

Reproducibility: all 88 fits were checked against their saved predictions. Yorùbá was also reproduced by the user in Colab; the new three-language runs await the user's own reproduction.


### Combined abstract

Learning with Less: Comparing Word and Character Features for News Classification in Four Nigerian Languages

Does the choice between word and character features matter when labelled news data is limited? This study compares the two representations for Yorùbá, Igbo, Hausa, and Nigerian Pidgin using MasakhaNEWS. We train separate logistic regression classifiers for each language, using either word unigrams and bigrams or character sequences of three to five characters, represented with term frequency–inverse document frequency (TF-IDF).

After automated checks for empty text and overlapping articles, the retained training sets contain 1,433 Yorùbá, 1,295 Igbo, 2,216 Hausa, and 1,058 Nigerian Pidgin articles. We evaluate models at approximately 25%, 50%, and 100% of each language’s available training data. At the smaller budgets, five matched, class-stratified subsets are used for both representations; each full-data model is fitted once. Text features are learned only from the corresponding training subset. Evaluation uses fixed held-out test sets of 411, 361, 633, and 300 articles, respectively.

Character features achieve higher mean macro-F1 in all twelve language-budget comparisons, with advantages ranging from 1.11 to 3.08 points on a 0–100 scale. With full training data, character versus word scores are 87.90 versus 86.79 for Yorùbá, 72.63 versus 69.94 for Igbo, 86.67 versus 85.50 for Hausa, and 84.86 versus 82.75 for Nigerian Pidgin. Both approaches improve as training data increases in every language.

These findings support character TF-IDF as a useful baseline across the four language datasets under the tested settings. Dataset sizes and category sets differ, so results are interpreted within each language rather than used to rank languages. The study uses one corpus and fixed model settings; statistical significance, cross-domain generalisation, and comparisons with pretrained language models remain untested.
